<div style="padding: 35px 25px; background: linear-gradient(135deg, #fdfbf7 0%, #f5efe1 100%); border-left: 6px solid #d4af37; border-radius: 16px; border: 1px solid rgba(212, 175, 55, 0.25); box-shadow: 0 10px 25px rgba(212, 175, 55, 0.12); font-family: 'Georgia', 'Segoe UI', serif; text-align: left; position: relative;"><div style="display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px;"><span style="background: linear-gradient(90deg, #d4af37, #aa7c11); color: #ffffff; padding: 5px 14px; border-radius: 20px; font-family: sans-serif; font-size: 0.75rem; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; display: inline-block;">💼 Career Pipeline</span><span style="font-family: 'Brush Script MT', 'cursive', sans-serif; color: #aa7c11; font-size: 1.5rem; font-weight: 500;">By Poonam Rajput ✨</span></div><h1 style="color: #3a2e1d; margin: 18px 0 6px 0; font-size: 2.4rem; font-weight: 700; letter-spacing: -0.5px;">Web Scraping <span style="color: #aa7c11; font-style: italic;">Indeed Jobs</span></h1><p style="color: #6e5d47; margin: 0 0 18px 0; font-size: 0.95rem; font-family: sans-serif; font-weight: 400; opacity: 0.85;">Extracting Live Positions. Tracking Employment Trends.</p><div style="height: 1px; background: linear-gradient(90deg, rgba(212, 175, 55, 0.3), transparent); margin: 15px 0;"></div><div style="display: flex; gap: 20px; flex-wrap: wrap; font-family: sans-serif; font-size: 0.85rem; color: #5c4d39; font-weight: 500;"><span>🎯 <b>Target:</b> Job Cards &amp; Salaries</span><span>⚙️ <b>Engine:</b> Selenium WebDriver</span><span>📊 <b>Output:</b> Pandas DataFrame</span></div></div>

<div style="padding: 15px 20px; background: linear-gradient(135deg, #fdfbf7 0%, #f5efe1 100%); border-left: 5px solid #d4af37; border-radius: 8px; border: 1px solid rgba(212, 175, 55, 0.2); font-family: 'Segoe UI', system-ui, sans-serif; text-align: left; display: flex; align-items: center; gap: 10px;"><span style="font-size: 1.2rem;">📦</span><span style="color: #3a2e1d; font-size: 1.1rem; font-weight: 700; letter-spacing: -0.3px;">Importing <span style="color: #aa7c11; font-style: italic;">Libraries</span></span></div>

In [14]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


<div style="padding: 15px 20px; background: linear-gradient(135deg, #fdfbf7 0%, #f5efe1 100%); border-left: 5px solid #d4af37; border-radius: 8px; border: 1px solid rgba(212, 175, 55, 0.2); font-family: 'Segoe UI', system-ui, sans-serif; text-align: left; display: flex; align-items: center; gap: 10px;"><span style="font-size: 1.2rem;">📦</span><span style="color: #3a2e1d; font-size: 1.1rem; font-weight: 700; letter-spacing: -0.3px;">Extract <span style="color: #aa7c11; font-style: italic;">Data</span></span></div>

In [15]:


def scrape_indeed_jobs(job_title="Python Developer", location="Remote"):
    """
    Robust Indeed Job Scraper with fault-tolerant individual column tracking.
    """
    options = Options()
    
    # Normal window layout helps glide past Cloudflare perimeter filters automatically.
    # If this runs successfully on your machine, you can change to options.add_argument("--headless=new")
    options.add_argument("--window-size=1440,900")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    # Inject real desktop environment profiles to decouple default automated signatures
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    # Clear automation navigator variables
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    
    parsed_jobs = []
    
    try:
        # Build structured query parameters
        formatted_title = job_title.replace(" ", "+")
        formatted_loc = location.replace(" ", "+")
        target_url = f"https://www.indeed.com/jobs?q={formatted_title}&l={formatted_loc}"
        
        print(f"[INFO] Initializing browser target sequence to: {target_url}")
        driver.get(target_url)
        
        # Give JS dynamic components ample time to render listing columns
        time.sleep(5)
        
        # Explicit Wait: Confirm job container nodes are injected into viewport DOM
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.job_seen_beacon")))
        
        job_cards = driver.find_elements(By.CSS_SELECTOR, "div.job_seen_beacon")
        print(f"[SUCCESS] Discovered {len(job_cards)} job cards available on page.")
        
        rank_counter = 1
        for card in job_cards:
            if rank_counter > 15:  # Cap pull output to the top 15 results
                break
                
            # Initialize structured mapping block defaults for this job row
            job_record = {
                "Rank": rank_counter,
                "Title": "Not Disclosed",
                "Company": "Confidential Employer",
                "Location": "Not Specified",
                "Salary": "Not Disclosed",
                "URL": "None"
            }
            
            # --- Field 1: Job Title Extraction ---
            try:
                title_el = card.find_element(By.CSS_SELECTOR, "h2.jobTitle span[id^='jobTitle-']")
                job_record["Title"] = title_el.text.strip()
            except:
                try:
                    # Fallback backup selector path for varying A/B test UI models
                    job_record["Title"] = card.find_element(By.CSS_SELECTOR, "h2.jobTitle").text.strip()
                except:
                    pass
            
            # --- Field 2: Company Name Extraction ---
            try:
                # Target primary custom structural test attribute tag
                company_el = card.find_element(By.CSS_SELECTOR, "[data-testid='company-name']")
                job_record["Company"] = company_el.text.strip()
            except:
                try:
                    job_record["Company"] = card.find_element(By.CSS_SELECTOR, "span.css-63beeb").text.strip()
                except:
                    pass
                    
            # --- Field 3: Job Location Extraction ---
            try:
                location_el = card.find_element(By.CSS_SELECTOR, "[data-testid='text-location']")
                job_record["Location"] = location_el.text.strip()
            except:
                try:
                    job_record["Location"] = card.find_element(By.CSS_SELECTOR, "div.css-1p0nf51").text.strip()
                except:
                    pass

            # --- Field 4: Estimated Salary Extraction ---
            salary_selectors = [
                "div.metadata.salarySnippet-container",
                "div.metadata.estimatedSalary-container",
                "div.salary-snippet-container",
                "div.css-18v9vux"
            ]
            for selector in salary_selectors:
                try:
                    salary_el = card.find_element(By.CSS_SELECTOR, selector)
                    job_record["Salary"] = salary_el.text.strip()
                    break # Break custom tracking cascade once text content string captures cleanly
                except:
                    continue

            # --- Field 5: Core Listing Link Extraction ---
            try:
                link_el = card.find_element(By.CSS_SELECTOR, "a.jcs-JobTitle")
                job_record["URL"] = link_el.get_attribute("href")
            except:
                pass
            
            # Save whatever content blocks successfully evaluated
            parsed_jobs.append(job_record)
            rank_counter += 1

    except Exception as e:
        print(f"[CRITICAL ERROR] Core pipeline pipeline failure: {e}")
        
    finally:
        print("[INFO] Shutting down selenium tracking engine threads safely...")
        driver.quit()
        
    return parsed_jobs




In [16]:
scraped_data = scrape_indeed_jobs(job_title="Python Developer", location="Remote")


[INFO] Initializing browser target sequence to: https://www.indeed.com/jobs?q=Python+Developer&l=Remote
[SUCCESS] Discovered 16 job cards available on page.
[INFO] Shutting down selenium tracking engine threads safely...


<div style="padding: 15px 20px; background: linear-gradient(135deg, #fdfbf7 0%, #f5efe1 100%); border-left: 5px solid #d4af37; border-radius: 8px; border: 1px solid rgba(212, 175, 55, 0.2); font-family: 'Segoe UI', system-ui, sans-serif; text-align: left; display: flex; align-items: center; gap: 10px;"><span style="font-size: 1.2rem;">📦</span><span style="color: #3a2e1d; font-size: 1.1rem; font-weight: 700; letter-spacing: -0.3px;">Converting to Data Frame<span style="color: #aa7c11; font-style: italic;"></span></span></div>

In [17]:
df = pd.DataFrame(scraped_data)


In [18]:
df

,Rank,Title,Company,Location,Salary,URL
0,1,Not Disclosed,Amentum,Hybrid work,Not Disclosed,https://www.indeed.com/rc/clk?jk=b33f7fd4cb769...
1,2,Not Disclosed,Intone Networks,Remote,Not Disclosed,https://www.indeed.com/rc/clk?jk=48d4e37895373...
2,3,Not Disclosed,Robotics Technologies,"Remote in Asheville, NC",Not Disclosed,https://www.indeed.com/rc/clk?jk=cdd02ac49f924...
3,4,Not Disclosed,CEDENT,Remote,Not Disclosed,https://www.indeed.com/rc/clk?jk=67fdeec7c9449...
4,5,Not Disclosed,Netflix,Remote,Not Disclosed,https://www.indeed.com/rc/clk?jk=57c55b04403cc...
5,6,Not Disclosed,Home Depot / THD,"Remote in Atlanta, GA",Not Disclosed,https://www.indeed.com/rc/clk?jk=51aa93a97bd82...
6,7,Not Disclosed,3Pillar,Remote,Not Disclosed,https://www.indeed.com/rc/clk?jk=c3a6065cebb14...
7,8,Not Disclosed,Barclays,"Remote in Whippany, NJ",Not Disclosed,https://www.indeed.com/rc/clk?jk=1e0d243f361bd...
8,9,Not Disclosed,,,Not Disclosed,https://www.indeed.com/viewjob?jk=123456789abc...
9,10,Not Disclosed,General Motors,"Remote in Austin, TX",Not Disclosed,https://www.indeed.com/rc/clk?jk=e8498c08bb6af...
